# Import Library

In [1]:
import emoji
import pandas as pd
import re
import ast
import numpy as np
import string

# Import Dataset

In [2]:
df = pd.read_csv("data_with_true_label_fixed.csv", encoding='utf-8-sig')
df

,post_shortcode,post_date,commenter_username,comment_text,comment_likes,true_aspect_1,true_aspect_2,true_sentiment,type
0,DFKccJJPeW_,23/01/2025 09:32,tiara_180319,Pemasangan baru tidak ada kelanjutan setelah s...,0,Pemasangan,Pelayanan,Negatif,Pernyataan
1,DFKccJJPeW_,23/01/2025 09:32,dshellysss,Sama naik semua ini juga komplen..biasanya cum...,0,Harga,Harga,Negatif,Pernyataan
2,DFKccJJPeW_,23/01/2025 09:32,shicya_cute,Min kalau puteran PDAM dol itu sya ngadu kesia...,0,Meteran (Macet/Bermasalah),Meteran (Macet/Bermasalah),Netral,Pertanyaan
3,DFKccJJPeW_,23/01/2025 09:32,iirmawulan,Denda PDAM telat ngalah2in pinjol 50k jadi 150...,1,Harga,Harga,Negatif,Pernyataan
4,DFKccJJPeW_,23/01/2025 09:32,aini_al_aydrus,Air ya klo pagi dan jam 3 sore kenapa selalu mati,1,Air Tidak Mengalir,Air Tidak Mengalir,Negatif,Pertanyaan
...,...,...,...,...,...,...,...,...,...
743,DGe2WlxPbvn,25/02/2025 04:15,mell0dz,gunung anyar tambak apa juga terdampak yaa kok...,1,Air Tidak Mengalir,Air Tidak Mengalir,Netral,Pertanyaan
744,DGe2WlxPbvn,25/02/2025 04:15,dodo_hd12,"Hasilnya gimana, min @pdamsuryasembada ??",1,Lainnya,Lainnya,Netral,Pertanyaan
745,DGcY2FIPTcc,24/02/2025 05:20,adiguna2022,min. daerah sidodadi ada pipa besar bocor. moh...,1,Kebocoran,Kebocoran,Negatif,Pernyataan
746,DGZ7MuDvXk3,23/02/2025 06:22,nikelu22,Min. Ini pdam saya mati sekitar tgl segini. Sa...,0,Air Tidak Mengalir,Pelayanan,Netral,Pertanyaan


# Removing Duplicates

In [3]:
df.duplicated().sum()

np.int64(2)

In [4]:
df = df.drop_duplicates()
df = df.reset_index(drop=True)
df.duplicated().sum()

np.int64(0)

# Preparing Emoji Dataset

In [5]:
df_emoji = pd.read_csv('emoji.csv', index_col=0)

In [6]:
# Ganti spasi dengan underscore pada kolom 'name'
df_emoji['name'] = df_emoji['name'].str.replace(' ', '_')
df_emoji['name'] = df_emoji['name'].str.replace('-', '_')
df_emoji['name'] = df_emoji['name'].str.replace('⊛_', '')
df_emoji['name'] = df_emoji['name'].str.replace('"', '')
df_emoji['name'] = df_emoji['name'].str.replace(':', '')
df_emoji['name'] = df_emoji['name'].str.replace(',', '')
df_emoji['name'] = df_emoji['name'].str.replace("'", "")
# Tampilkan 5 baris pertama untuk memastikan hasilnya
print(df_emoji.head())

                              name
0                    grinning_face
1      grinning_face_with_big_eyes
2  grinning_face_with_smiling_eyes
3   beaming_face_with_smiling_eyes
4          grinning_squinting_face


In [7]:
df_emoji.to_csv('emoji_underscore.csv', index=True)

In [8]:
df_emoji

,name
0,grinning_face
1,grinning_face_with_big_eyes
2,grinning_face_with_smiling_eyes
3,beaming_face_with_smiling_eyes
4,grinning_squinting_face
...,...
1811,flag_Zambia
1812,flag_Zimbabwe
1813,flag_England
1814,flag_Scotland


# Cleansing

In [9]:
# --- 1. Baca daftar emoji dari file emoji_underscore.csv ---
emoji_df = pd.read_csv('emoji_underscore.csv')
emoji_df['name'] = emoji_df['name'].astype(str).str.strip()

# Buat pola regex berdasarkan nama emoji (format :emoji_name:)
emoji_names = [re.escape(name) for name in emoji_df['name'] if name]
emoji_pattern = re.compile(r':(' + '|'.join(emoji_names) + r'):', flags=re.IGNORECASE)


# --- 2. Fungsi pembersihan teks ---
def clean_text(text):
    if pd.isna(text):
        return ''
    
    text = str(text).lower()

    # Hapus emoji berdasarkan daftar emoji
    text = emoji_pattern.sub('', text)

    # Hapus URL
    text = re.sub(r'http\S+|www.\S+', '', text)
    
    # Hapus tanda hubung "-"
    text = re.sub('-', ' ', text)
    
    # Hapus mention dan hashtag
    text = re.sub(r'@\w+|#\w+', '', text)
    
    # Hapus angka dan simbol non-huruf
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


# Terapkan fungsi clean_text ke kolom comment_text
df['cleaned_text'] = df['comment_text'].apply(clean_text)

# --- 4. Hapus baris yang kosong atau hanya berisi emoji (setelah dibersihkan jadi kosong) ---
before = len(df)
df = df[df['cleaned_text'].str.strip().astype(bool)]
after = len(df)

print(f"🧹 {before - after} baris dihapus karena hanya berisi emoji atau kosong.")

🧹 55 baris dihapus karena hanya berisi emoji atau kosong.


In [10]:
print(df['cleaned_text'].to_markdown())

|     | cleaned_text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|----:|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Stopword Removal + Normalization

In [11]:
import nltk
from nltk.tokenize import word_tokenize

# Pastikan tokenizer tersedia
nltk.download('punkt', quiet=True)

# --- Load stopwords ---
with open("combined_stop_words.txt", "r", encoding="utf-8") as f:
    stop_words = f.read().splitlines()

# --- Load slang dictionary ---
with open("update_combined_slang_words.txt", "r", encoding="utf-8") as f:
    slang_words = ast.literal_eval(f.read())


# --- Fungsi Stopword Removal + Slang Formalization ---
def formalize_text(text):

    if pd.isna(text) or not isinstance(text, str):
        return ""

    # Tokenisasi sementara
    tokens = word_tokenize(text.lower())

    # Slang formalization
    normalized_tokens = [slang_words.get(w, w) for w in tokens]

    # Stopword removal
    filtered_tokens = [
        w for w in normalized_tokens
        if w not in stop_words and w not in string.punctuation and w.strip()
    ]

    # Gabungkan kembali menjadi teks
    return " ".join(filtered_tokens)


# Terapkan pada cleaned_text
df['normalized_text'] = df['cleaned_text'].apply(formalize_text)

print(df[['cleaned_text', 'normalized_text']].head().to_markdown())

|    | cleaned_text                                                                                                                 | normalized_text                                                                                                             |
|---:|:-----------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------|
|  0 | pemasangan baru tidak ada kelanjutan setelah surveykatanya bisa komplain cs lewat wasedangkan di wa gak ada tanggapan        | pemasangan tidak ada kelanjutan survey katanya komplain pelayanan pelanggan whatsapp sedangkan whatsapp tidak ada tanggapan |
|  1 | sama naik semua ini juga komplenbiasanya cuma an sekrang anduh semua serba membayar to jadi g percaya sama pemerintah begini | naik komplain biasanya an sekarang aduh serba membayar kan ga percaya pemerintah      

# Drop Empty row

In [12]:
# --- Hapus baris dengan normalized_text kosong ---
before = len(df)

df = df[
    df['normalized_text'].notna() & 
    (df['normalized_text'].astype(str).str.strip() != "")
]

after = len(df)

print("✅ Baris kosong pada kolom 'normalized_text' berhasil dihapus.")
print(f"🗑️ Total baris dihapus: {before - after}")

# Simpan dataset
# df.to_csv("cleaned_normalized_dataset.csv", index=False, encoding='utf-8-sig')

✅ Baris kosong pada kolom 'normalized_text' berhasil dihapus.
🗑️ Total baris dihapus: 11


# Segmentasi Teks

In [13]:
import pandas as pd
import re

# =========================================================
# 1️⃣ LOAD LIST KONJUNGSI
# =========================================================

with open("augmentation_text_dict.txt", "r", encoding="utf-8") as f:
    conjunctions = [line.strip().lower() for line in f if line.strip()]

# contoh isi file
# sedang
# lalu
# tapi
# karena
# tetapi
# padahal
# akan


# =========================================================
# 2️⃣ FUNGSI SPLIT BERDASARKAN KONJUNGSI
# =========================================================

def split_by_conjunction(text):

    if pd.isna(text) or not isinstance(text, str):
        return pd.Series(["", ""])

    text_lower = text.lower()

    for conj in conjunctions:

        # cari konjungsi sebagai kata utuh
        pattern = r'\b' + re.escape(conj) + r'\b'
        match = re.search(pattern, text_lower)

        if match:
            idx = match.start()

            sentence_1 = text[:idx].strip()
            sentence_2 = text[idx:].strip()

            return pd.Series([sentence_1, sentence_2])

    # jika tidak ada konjungsi
    return pd.Series([text.strip(), ""])


# =========================================================
# 3️⃣ APLIKASI KE DATAFRAME
# =========================================================

df[['sentence_1', 'sentence_2']] = df['normalized_text'].apply(split_by_conjunction)


# =========================================================
# 4️⃣ HASIL
# =========================================================

print(df[['normalized_text', 'sentence_1', 'sentence_2']].head())

# simpan dataset
# df.to_csv("data_with_augmented_sentences.csv", index=False, encoding='utf-8-sig')

                                     normalized_text  \
0  pemasangan tidak ada kelanjutan survey katanya...   
1  naik komplain biasanya an sekarang aduh serba ...   
2            admin keran pdam rusak mengadu ke siapa   
3      denda pdam terlambat mengalahkan pinjol parah   
4                      air pagi jam sore selalu mati   

                                          sentence_1  \
0  pemasangan tidak ada kelanjutan survey katanya...   
1  naik komplain biasanya an sekarang aduh serba ...   
2            admin keran pdam rusak mengadu ke siapa   
3      denda pdam terlambat mengalahkan pinjol parah   
4                      air pagi jam sore selalu mati   

                               sentence_2  
0  sedangkan whatsapp tidak ada tanggapan  
1                                          
2                                          
3                                          
4                                          


# Tokenization

In [14]:
from nltk.tokenize import word_tokenize

# =========================================================
# FUNGSI TOKENIZATION
# =========================================================

def tokenize_text(text):
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        return []
    return word_tokenize(text)


# =========================================================
# TOKENIZATION PADA 3 KOLOM
# =========================================================

df['tokens_normalized'] = df['normalized_text'].apply(tokenize_text)

df['tokens_sentence_1'] = df['sentence_1'].apply(tokenize_text)

df['tokens_sentence_2'] = df['sentence_2'].apply(tokenize_text)


# =========================================================
# HASIL
# =========================================================

print(
    df[
        [
            'normalized_text',
            'sentence_1',
            'sentence_2',
            'tokens_normalized',
            'tokens_sentence_1',
            'tokens_sentence_2'
        ]
    ].head().to_markdown()
)

|    | normalized_text                                                                                                             | sentence_1                                                                           | sentence_2                             | tokens_normalized                                                                                                                                                         | tokens_sentence_1                                                                                                     | tokens_sentence_2                                      |
|---:|:----------------------------------------------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------|:---------------------------------------|:---------------------------------------------------------------------------------------------------------------------------------

# Stemming

In [15]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# =========================================================
# 1️⃣ INISIALISASI STEMMER
# =========================================================

factory = StemmerFactory()
stemmer = factory.create_stemmer()


# =========================================================
# 2️⃣ FUNGSI STEMMING
# =========================================================

def stem_tokens(tokens):

    if isinstance(tokens, list):
        text = " ".join(tokens)
        return stemmer.stem(text)

    elif isinstance(tokens, str):
        return stemmer.stem(tokens)

    else:
        return ""


# =========================================================
# 3️⃣ STEMMING PADA SETIAP KOLOM TOKEN
# =========================================================

df['stemmed_normalized'] = df['tokens_normalized'].apply(stem_tokens)

df['stemmed_sentence_1'] = df['tokens_sentence_1'].apply(stem_tokens)

df['stemmed_sentence_2'] = df['tokens_sentence_2'].apply(stem_tokens)


# =========================================================
# 4️⃣ HASIL
# =========================================================

print(
    df[
        [
            'tokens_normalized',
            'tokens_sentence_1',
            'tokens_sentence_2',
            'stemmed_normalized',
            'stemmed_sentence_1',
            'stemmed_sentence_2'
        ]
    ].head().to_markdown()
)


# =========================================================
# 5️⃣ SIMPAN DATASET
# =========================================================

# df[
#     [
#         'post_shortcode',
#         'commenter_username',
#         'comment_text',
#         'cleaned_text',
#         'normalized_text',
#         'true_aspect_1',
#         'true_aspect_2',
#         'true_sentiment',
#         'type',
#         'sentence_1',
#         'sentence_2',
#         'tokens_normalized',
#         'tokens_sentence_1',
#         'tokens_sentence_2',
#         'stemmed_normalized',
#         'stemmed_sentence_1',
#         'stemmed_sentence_2'
#     ]
# ].to_csv("final_stemmed_dataset.csv", index=False, encoding='utf-8-sig')

|    | tokens_normalized                                                                                                                                                         | tokens_sentence_1                                                                                                     | tokens_sentence_2                                      | stemmed_normalized                                                                                    | stemmed_sentence_1                                                       | stemmed_sentence_2                |
|---:|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------|:-------------------------------------------------------|:-------------------------------------------------------------------------

# Train: Not Segmented Data | Test: 100% Data

In [16]:
df_test = df

# Mengambil data yang nilai sentence_2 kosong (NaN atau string kosong)
df_train = df[df['sentence_2'].isna() | (df['sentence_2'].str.strip() == '')]
df_train = df_train.drop(columns=['sentence_1', 'sentence_2', 'tokens_sentence_1', 'tokens_sentence_2', 'stemmed_sentence_1', 'stemmed_sentence_2'])
# Reset index agar rapi
df_train = df_train.reset_index(drop=True)

# Menampilkan hasil
print(df_train)
df_train.to_csv("train_data.csv", index=False, encoding='utf-8-sig')

    post_shortcode         post_date commenter_username  \
0      DFKccJJPeW_  23/01/2025 09:32         dshellysss   
1      DFKccJJPeW_  23/01/2025 09:32        shicya_cute   
2      DFKccJJPeW_  23/01/2025 09:32         iirmawulan   
3      DFKccJJPeW_  23/01/2025 09:32     aini_al_aydrus   
4      DFKccJJPeW_  23/01/2025 09:32          firdaulyh   
..             ...               ...                ...   
568    DGe2WlxPbvn  25/02/2025 04:15        pujianaanin   
569    DGe2WlxPbvn  25/02/2025 04:15            mell0dz   
570    DGe2WlxPbvn  25/02/2025 04:15          dodo_hd12   
571    DGcY2FIPTcc  24/02/2025 05:20        adiguna2022   
572    DGZYtHQz-lx  23/02/2025 01:19         e_noegroho   

                                          comment_text  comment_likes  \
0    Sama naik semua ini juga komplen..biasanya cum...              0   
1    Min kalau puteran PDAM dol itu sya ngadu kesia...              0   
2    Denda PDAM telat ngalah2in pinjol 50k jadi 150...              1   

# Feature Extraction & Multi Aspect Based Classification

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# 1️⃣ VECTORISASI TF-IDF (TRAIN DATA)
# Train menggunakan stemmed_normalized
# =========================================================

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=2
)

X_train = vectorizer.fit_transform(df_train['stemmed_normalized'])

train_aspects = df_train['true_aspect_1'].to_numpy()


# =========================================================
# 2️⃣ FUNGSI PREDIKSI ASPEK
# =========================================================

THRESHOLD = 0.3

def predict_aspect(text):

    if pd.isna(text) or str(text).strip() == "":
        return "Lainnya", 0

    X_test = vectorizer.transform([text])

    sims = cosine_similarity(X_test, X_train)[0]

    best_idx = sims.argmax()
    best_score = sims[best_idx]
    best_aspect = train_aspects[best_idx]

    if best_score < THRESHOLD:
        return "Lainnya", best_score

    return best_aspect, best_score


# =========================================================
# 3️⃣ MULTI-ASPECT CLASSIFICATION
# =========================================================

predicted_aspects = []
similarity_scores = []

for i,row in df_test.iterrows():

    text1 = row['stemmed_sentence_1']
    text2 = row['stemmed_sentence_2']

    # ----------------------------------
    # ASPECT KALIMAT 1
    # ----------------------------------
    aspect1, score1 = predict_aspect(text1)

    # ----------------------------------
    # ASPECT KALIMAT 2
    # ----------------------------------
    if pd.isna(text2) or str(text2).strip()=="":
        aspect2 = aspect1
        score2 = score1
    else:
        aspect2, score2 = predict_aspect(text2)

    # ----------------------------------
    # GABUNGKAN ASPEK (SELALU 2 ASPEK)
    # ----------------------------------
    final_aspect = f"{aspect1} | {aspect2}"

    final_score = max(score1,score2)

    predicted_aspects.append(final_aspect)
    similarity_scores.append(final_score)

    # =====================================================
    # DEBUG PRINT
    # =====================================================
    print("="*80)
    print("Index :",i)
    print("Sentence 1 :",text1)
    print("Sentence 2 :",text2)
    print("Predicted :",final_aspect)
    print("Score 1 :",round(score1,4))
    print("Score 2 :",round(score2,4))
    print("="*80,"\n")


# =========================================================
# 4️⃣ SIMPAN HASIL
# =========================================================

df_test['predicted_aspects'] = predicted_aspects
df_test['similarity_scores'] = similarity_scores


output_cols = [
'commenter_username',
'comment_text',
'cleaned_text',
'normalized_text',
'stemmed_sentence_1',
'stemmed_sentence_2',
'true_aspect_1',
'true_aspect_2',
'true_sentiment',
'type',
'predicted_aspects',
'similarity_scores'
]


df_test[output_cols].to_csv(
"trial_multi_aspect_classification_result.csv",
index=False,
encoding="utf-8-sig"
)

print("✅ Multi-aspect classification selesai")

Index : 0
Sentence 1 : pasang tidak ada lanjut survey kata komplain layan langgan whatsapp
Sentence 2 : sedang whatsapp tidak ada tanggap
Predicted : Pelayanan | Pelayanan
Score 1 : 0.4098
Score 2 : 0.4996

Index : 1
Sentence 1 : naik komplain biasa an sekarang aduh serba bayar kan ga percaya perintah
Sentence 2 : 
Predicted : Harga | Harga
Score 1 : 1.0
Score 2 : 1.0

Index : 2
Sentence 1 : admin keran pdam rusak adu ke siapa
Sentence 2 : 
Predicted : Meteran (Macet/Bermasalah) | Meteran (Macet/Bermasalah)
Score 1 : 1.0
Score 2 : 1.0

Index : 3
Sentence 1 : denda pdam lambat kalah pinjol parah
Sentence 2 : 
Predicted : Harga | Harga
Score 1 : 1.0
Score 2 : 1.0

Index : 4
Sentence 1 : air pagi jam sore selalu mati
Sentence 2 : 
Predicted : Air Tidak Mengalir | Air Tidak Mengalir
Score 1 : 1.0
Score 2 : 1.0

Index : 5
Sentence 1 : si sudah hubung layan adu sudah hampir minggu tidak ada tindak
Sentence 2 : 
Predicted : Pelayanan | Pelayanan
Score 1 : 1.0
Score 2 : 1.0

Index : 6
Sentence

# Multi Aspect Based Evaluation

In [18]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# =========================================================
# 1️⃣ BENTUK SET TRUE ASPECT (MULTI)
# =========================================================

def build_true_aspect_set(row):

    aspects = set()

    if pd.notna(row['true_aspect_1']) and row['true_aspect_1'] != 'Lainnya':
        aspects.add(row['true_aspect_1'])

    if pd.notna(row['true_aspect_2']) and row['true_aspect_2'] != 'Lainnya':
        aspects.add(row['true_aspect_2'])

    if len(aspects) == 0:
        aspects.add('Lainnya')

    return aspects


df_test['true_aspects_set'] = df_test.apply(build_true_aspect_set, axis=1)


# =========================================================
# 2️⃣ BENTUK SET PREDICTED ASPECT
# =========================================================

def build_predicted_aspect_set(pred):

    if pd.isna(pred) or pred == 'Lainnya':
        return {'Lainnya'}

    return set([a.strip() for a in pred.split('|')])


df_test['predicted_aspects_set'] = df_test['predicted_aspects'].apply(build_predicted_aspect_set)


print(f"Total Data Train : {len(df_train)}")
print(f"Total Data Test  : {len(df_test)}")


# =========================================================
# 3️⃣ EVALUASI MULTI-ASPECT
# =========================================================

# Exact Match
df_test['exact_match'] = (
    df_test['true_aspects_set'] == df_test['predicted_aspects_set']
)

exact_accuracy = df_test['exact_match'].mean()


# Partial Match
df_test['partial_match'] = df_test.apply(
    lambda x: len(x['true_aspects_set'].intersection(x['predicted_aspects_set'])) > 0,
    axis=1
)

partial_accuracy = df_test['partial_match'].mean()


print("\n==============================")
print("🎯 EVALUASI MULTI-ASPECT")
print("==============================")
print(f"Exact Match Accuracy   : {exact_accuracy:.4f}")
print(f"Partial Match Accuracy : {partial_accuracy:.4f}")


# =========================================================
# 4️⃣ STATISTIK PREDICTED ASPECT
# =========================================================

aspect_stats = (
    df_test
    .groupby('predicted_aspects')
    .agg(
        Jumlah_Kemunculan=('predicted_aspects','count'),
        Rata_rata_Similarity=('similarity_scores','mean')
    )
    .reset_index()
    .sort_values('Jumlah_Kemunculan', ascending=False)
)

print("\n===========================================")
print("📊 Statistik Predicted Aspect (Test Data)")
print("===========================================")
print(aspect_stats.to_markdown(index=False))


# =========================================================
# 5️⃣ DISTRIBUSI TRUE ASPECT (MULTI)
# =========================================================

true_aspect_exploded = (
    df_test['true_aspects_set']
    .apply(list)
    .explode()
    .value_counts()
    .reset_index()
)

true_aspect_exploded.columns = ['Aspek','Jumlah']


print("\n==================================================")
print("📌 Distribusi True Aspect (Multi-Aspect)")
print("==================================================")
print(true_aspect_exploded.to_markdown(index=False))


# =========================================================
# 6️⃣ TOTAL SALAH & BENAR
# =========================================================

total_correct = df_test['exact_match'].sum()
total_wrong   = len(df_test) - total_correct


print("\n===========================================")
print("❌ HASIL KLASIFIKASI MULTI-ASPECT")
print("===========================================")
print(f"Total Data Test        : {len(df_test)}")
print(f"Prediksi Benar (Exact) : {total_correct}")
print(f"Prediksi Salah         : {total_wrong}")
print(f"Error Rate             : {total_wrong/len(df_test)*100:.2f}%")


# =========================================================
# 7️⃣ DATA SALAH PREDIKSI
# =========================================================

misclassified_df = df_test.loc[
    ~df_test['exact_match'],
    [
        'commenter_username',
        'comment_text',
        'stemmed_normalized',
        'stemmed_sentence_1',
        'stemmed_sentence_2',
        'true_aspect_1',
        'true_aspect_2',
        'true_aspects_set',
        'predicted_aspects_set',
        'similarity_scores'
    ]
]

misclassified_df.to_csv(
    "trial_misclassified_multi_aspect_data.csv",
    index=False,
    encoding='utf-8-sig'
)

print("\n✅ Evaluasi multi-aspect selesai")

Total Data Train : 573
Total Data Test  : 680

🎯 EVALUASI MULTI-ASPECT
Exact Match Accuracy   : 0.8706
Partial Match Accuracy : 0.9618

📊 Statistik Predicted Aspect (Test Data)
| predicted_aspects                                       |   Jumlah_Kemunculan |   Rata_rata_Similarity |
|:--------------------------------------------------------|--------------------:|-----------------------:|
| Lainnya | Lainnya                                       |                 246 |               0.920396 |
| Air Tidak Mengalir | Air Tidak Mengalir                 |                 200 |               0.979812 |
| Pelayanan | Pelayanan                                   |                  46 |               0.958438 |
| Air Kotor/Bau | Air Kotor/Bau                           |                  36 |               0.983815 |
| Harga | Harga                                           |                  34 |               0.990544 |
| Meteran (Macet/Bermasalah) | Meteran (Macet/Bermasalah) |               